In [ ]:
# ==============================================================================
# CELL 1a: Feature extraction
#
# Extracts MFCC + LFCC + Chroma from:
#   Dataset A: FoR-norm  (for-norm/for-norm/[training|validation|testing])
#   Dataset B: WaveFake  (ljspeech_* vocoders, real = LJSpeech originals)
#
# Output structure (one npy per array, same as NB1 T3 cache):
#   /kaggle/working/pretrain_cache/
#     for_{train|val|test}_{mfcc|lfcc|chroma|labels}.npy
#     wf_{train|val|test}_{mfcc|lfcc|chroma|labels}.npy
#
# Labels: 0=real, 1=fake  (binary, no half-truth during pretraining)
# Temporal targets: not extracted (all -1.0 sentinel during pretraining)
#
# Disk budget:
#   FoR-norm train: 53,868 samples × 92 bytes = 4.96 GB
#   FoR-norm val:   10,798 samples × 92 bytes = 0.99 GB
#   FoR-norm test:   4,634 samples × 92 bytes = 0.43 GB
#   WaveFake train: ~74,000 samples × 92 bytes = 6.81 GB
#   WaveFake val:   ~10,000 samples × 92 bytes = 0.92 GB
#   WaveFake test:  ~8,300  samples × 92 bytes = 0.76 GB
#   Total: ~14.9 GB — extract one dataset at a time, upload, delete
# ==============================================================================

import gc, os, subprocess, librosa
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fftpack import dct

WORK_DIR      = Path("/kaggle/working")
CACHE_DIR     = WORK_DIR / "pretrain_cache"
FOR_BASE      = Path("/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm")
WF_BASE       = Path("/kaggle/input/datasets/walimuhammadahmad/fakeaudio/generated_audio")

SR=16000; DURATION=4.0; HOP=256; N_FFT=512
N_MFCC=40; N_LFCC=40; N_CHROMA=12
N_FRAMES = int(SR * DURATION / HOP) + 1   # 251

CACHE_DIR.mkdir(exist_ok=True)

# ── Helpers ─────────────────────────────────────────────────────

def disk():
    r = subprocess.run(['df','-h','/kaggle/working'], capture_output=True, text=True)
    line = [l for l in r.stdout.split('\n') if 'kaggle' in l]
    return line[0].split()[2] + "/" + line[0].split()[1] if line else "?"

def compute_lfcc(y, sr=16000, n_lfcc=40, n_fft=512, hop=256, n_filters=128):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    centers = np.linspace(0, freqs[-1], n_filters + 2)
    fb = np.zeros((n_filters, len(freqs)))
    for i in range(n_filters):
        fl, fc, fr = centers[i], centers[i+1], centers[i+2]
        for j, f in enumerate(freqs):
            if fl <= f <= fc:  fb[i,j] = (f-fl)/(fc-fl+1e-10)
            elif fc < f <= fr: fb[i,j] = (fr-f)/(fr-fc+1e-10)
    return dct(np.log(np.dot(fb, S)+1e-10), type=2, axis=0, norm="ortho")[:n_lfcc]

def extract_features(path, offset=0.0):
    try:
        y, _ = librosa.load(path, sr=SR, offset=offset, duration=DURATION)
        target = int(SR * DURATION)
        y = np.pad(y, (0, max(0, target-len(y))), mode='edge')[:target]
        def fix(f):
            if f.shape[1] < N_FRAMES:
                return np.pad(f,((0,0),(0,N_FRAMES-f.shape[1])),mode='edge')
            return f[:, :N_FRAMES]
        return (fix(librosa.feature.mfcc(y=y,sr=SR,n_mfcc=N_MFCC,
                    n_fft=N_FFT,hop_length=HOP)).astype(np.float32),
                fix(compute_lfcc(y,sr=SR)).astype(np.float32),
                fix(librosa.feature.chroma_stft(y=y,sr=SR,
                    n_fft=N_FFT,hop_length=HOP)).astype(np.float32))
    except Exception as e:
        print(f"  [warn] {Path(path).name}: {e}")
        return (np.zeros((N_MFCC,N_FRAMES),np.float32),
                np.zeros((N_LFCC,N_FRAMES),np.float32),
                np.zeros((N_CHROMA,N_FRAMES),np.float32))

def extract_binary_split(files_real, files_fake, prefix, tag):
    """
    Extract features for one split of a binary dataset.
    prefix: cache filename prefix e.g. 'for_train'
    tag: display name
    Saves: {prefix}_mfcc.npy, _lfcc.npy, _chroma.npy, _labels.npy
    Skips if all 4 files already exist.
    """
    keys = ["mfcc","lfcc","chroma","labels"]
    paths = {k: CACHE_DIR / f"{prefix}_{k}.npy" for k in keys}
    if all(p.exists() for p in paths.values()):
        arr = np.load(paths["labels"], mmap_mode='r')
        print(f"  {tag}: already done ({len(arr):,} samples) — skipping")
        return

    all_files = [(f, 0) for f in files_real] + [(f, 1) for f in files_fake]
    n = len(all_files)
    print(f"  {tag}: {len(files_real):,} real + {len(files_fake):,} fake = {n:,} total")
    print(f"    disk before: {disk()}")

    f_mfcc   = np.lib.format.open_memmap(paths["mfcc"],   mode='w+',
                 dtype=np.float32, shape=(n, N_MFCC,   N_FRAMES))
    f_lfcc   = np.lib.format.open_memmap(paths["lfcc"],   mode='w+',
                 dtype=np.float32, shape=(n, N_LFCC,   N_FRAMES))
    f_chroma = np.lib.format.open_memmap(paths["chroma"], mode='w+',
                 dtype=np.float32, shape=(n, N_CHROMA, N_FRAMES))
    f_labels = np.lib.format.open_memmap(paths["labels"], mode='w+',
                 dtype=np.int64,   shape=(n,))

    errors = 0
    for i, (fpath, label) in enumerate(tqdm(all_files, desc=f"    {tag}")):
        mfcc, lfcc, chroma = extract_features(fpath)
        if np.all(mfcc == 0): errors += 1
        f_mfcc[i]=mfcc; f_lfcc[i]=lfcc; f_chroma[i]=chroma; f_labels[i]=label

    for mm in [f_mfcc, f_lfcc, f_chroma, f_labels]:
        mm.flush(); del mm
    gc.collect()
    if errors: print(f"    [warn] {errors} extraction errors (zero-filled)")
    print(f"    disk after: {disk()}")


# ==============================================================================
# DATASET A: FoR-norm
# Structure: for-norm/[training|validation|testing]/[real|fake]/*.wav
# Use: training -> pretrain train, validation -> pretrain val, testing -> pretrain test
# ==============================================================================
print("="*65)
print("DATASET A: FoR-norm")
print("="*65)

FOR_SPLIT_MAP = {
    "train": "training",
    "val":   "validation",
    "test":  "testing",
}

for our_split, for_split in FOR_SPLIT_MAP.items():
    sp = FOR_BASE / for_split
    real_files = sorted((sp / "real").glob("*.wav"))
    fake_files = sorted((sp / "fake").glob("*.wav"))
    extract_binary_split(real_files, fake_files,
                         f"for_{our_split}", f"FoR-norm/{for_split}")

print(f"\nFoR-norm done. disk: {disk()}")
print(">>> Upload pretrain_cache/for_*.npy to Kaggle input dataset now")
print(">>> Then delete them to free disk before extracting WaveFake")
print()

# Uncomment after uploading FoR features:
# for f in CACHE_DIR.glob("for_*.npy"):
#     f.unlink()
#     print(f"Deleted {f.name}")


# ==============================================================================
# DATASET B: WaveFake (ljspeech vocoders only)
# Real audio: LJSpeech originals (ljspeech_waveglow contains originals)
# Fake audio: 6 ljspeech_* vocoder directories
# Split: 80/10/10 train/val/test (no pre-defined splits in WaveFake)
# ==============================================================================
print("="*65)
print("DATASET B: WaveFake (ljspeech vocoders)")
print("="*65)

# Real audio — LJSpeech originals
# ljspeech_waveglow contains the original LJ*.wav files (no _gen suffix)
LJ_REAL_DIR = WF_BASE / "ljspeech_waveglow"
real_files_all = sorted(f for f in LJ_REAL_DIR.glob("*.wav")
                        if not f.name.endswith("_gen.wav")
                        and not f.name.endswith("_generated.wav"))

# Fake audio — all 6 ljspeech vocoder dirs
WF_VOCODER_DIRS = [
    "ljspeech_full_band_melgan",
    "ljspeech_hifiGAN",
    "ljspeech_melgan",
    "ljspeech_melgan_large",
    "ljspeech_multi_band_melgan",
    "ljspeech_parallel_wavegan",
]
fake_files_all = []
for vdir in WF_VOCODER_DIRS:
    vpath = WF_BASE / vdir
    if vpath.exists():
        files = sorted(vpath.glob("*.wav"))
        fake_files_all.extend(files)
        print(f"  {vdir}: {len(files):,} files")

print(f"\n  Real (LJSpeech originals): {len(real_files_all):,}")
print(f"  Fake (all vocoders):       {len(fake_files_all):,}")

# 80/10/10 split — use same indices for real and fake independently
import random
random.seed(42)

def split_files(files, train_r=0.8, val_r=0.1):
    n = len(files)
    idx = list(range(n)); random.shuffle(idx)
    t = int(n * train_r); v = int(n * (train_r + val_r))
    return ([files[i] for i in idx[:t]],
            [files[i] for i in idx[t:v]],
            [files[i] for i in idx[v:]])

real_tr, real_val, real_te = split_files(real_files_all)
fake_tr, fake_val, fake_te = split_files(fake_files_all)

print(f"\n  Split sizes:")
print(f"    train: real={len(real_tr):,}  fake={len(fake_tr):,}")
print(f"    val:   real={len(real_val):,}  fake={len(fake_val):,}")
print(f"    test:  real={len(real_te):,}  fake={len(fake_te):,}")

for our_split, r_files, f_files in [
    ("train", real_tr,  fake_tr),
    ("val",   real_val, fake_val),
    ("test",  real_te,  fake_te),
]:
    extract_binary_split(r_files, f_files,
                         f"wf_{our_split}", f"WaveFake/{our_split}")

print(f"\nWaveFake done. disk: {disk()}")


# ==============================================================================
# VERIFY
# ==============================================================================
print()
print("="*65)
print("VERIFICATION")
print("="*65)
for prefix in ["for_train","for_val","for_test","wf_train","wf_val","wf_test"]:
    for key in ["mfcc","lfcc","chroma","labels"]:
        p = CACHE_DIR / f"{prefix}_{key}.npy"
        if p.exists():
            arr = np.load(p, mmap_mode='r')
            if key == "labels":
                r = int((arr==0).sum()); f = int((arr==1).sum())
                print(f"  {prefix}_{key}: {arr.shape}  real={r:,} fake={f:,}")
        else:
            print(f"  {prefix}_{key}: MISSING")

In [12]:
#============================================
#Clearing Directory
#============================================

from pathlib import Path

cache = Path("/kaggle/working/pretrain_cache")
if cache.exists():
    for f in sorted(cache.glob("*.npy")):
        print(f"  Deleting {f.name}  ({f.stat().st_size/1e6:.0f} MB)")
        f.unlink()
    try:
        cache.rmdir()
        print("Removed pretrain_cache/")
    except:
        print("pretrain_cache/ not empty — check remaining files")
else:
    print("pretrain_cache/ not found — already clean")

pretrain_cache/ not found — already clean


In [5]:
#============================================
# CELL 1b: ASVSpoof 2019 LA preprocessing
#============================================

import gc, subprocess, librosa
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fftpack import dct

WORK_DIR  = Path("/kaggle/working")
CACHE_DIR = WORK_DIR / "pretrain_cache"
ASV_BASE  = Path("/kaggle/input/datasets/piwoobeep/asvspoof2019la/LA")

SR=16000; DURATION=4.0; HOP=256; N_FFT=512
N_MFCC=40; N_LFCC=40; N_CHROMA=12
N_FRAMES = int(SR * DURATION / HOP) + 1

CACHE_DIR.mkdir(exist_ok=True)

# ── Protocol paths ────────────────────────────────────────────────────────────
PROTO_DIR = ASV_BASE / "ASVspoof2019_LA_cm_protocols"
PROTO = {
    "train": PROTO_DIR / "ASVspoof2019.LA.cm.train.trn.txt",
    "val":   PROTO_DIR / "ASVspoof2019.LA.cm.dev.trl.txt",
    "test":  PROTO_DIR / "ASVspoof2019.LA.cm.eval.trl.txt",
}
AUDIO_DIRS = {
    "train": ASV_BASE / "ASVspoof2019_LA_train" / "flac",
    "val":   ASV_BASE / "ASVspoof2019_LA_dev"   / "flac",
    "test":  ASV_BASE / "ASVspoof2019_LA_eval"  / "flac",
}

def disk():
    r = subprocess.run(['df','-h','/kaggle/working'], capture_output=True, text=True)
    line = [l for l in r.stdout.split('\n') if 'kaggle' in l]
    return line[0].split()[2] + "/" + line[0].split()[1] if line else "?"

def compute_lfcc(y, sr=16000, n_lfcc=40, n_fft=512, hop=256, n_filters=128):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    centers = np.linspace(0, freqs[-1], n_filters + 2)
    fb = np.zeros((n_filters, len(freqs)))
    for i in range(n_filters):
        fl, fc, fr = centers[i], centers[i+1], centers[i+2]
        for j, f in enumerate(freqs):
            if fl <= f <= fc:  fb[i,j] = (f-fl)/(fc-fl+1e-10)
            elif fc < f <= fr: fb[i,j] = (fr-f)/(fr-fc+1e-10)
    return dct(np.log(np.dot(fb, S)+1e-10), type=2, axis=0, norm="ortho")[:n_lfcc]

def extract_features(path, offset=0.0):
    try:
        y, _ = librosa.load(path, sr=SR, offset=offset, duration=DURATION)
        target = int(SR * DURATION)
        y = np.pad(y, (0, max(0, target-len(y))), mode='edge')[:target]
        def fix(f):
            if f.shape[1] < N_FRAMES:
                return np.pad(f,((0,0),(0,N_FRAMES-f.shape[1])),mode='edge')
            return f[:, :N_FRAMES]
        return (fix(librosa.feature.mfcc(y=y,sr=SR,n_mfcc=N_MFCC,
                    n_fft=N_FFT,hop_length=HOP)).astype(np.float32),
                fix(compute_lfcc(y)).astype(np.float32),
                fix(librosa.feature.chroma_stft(y=y,sr=SR,
                    n_fft=N_FFT,hop_length=HOP)).astype(np.float32))
    except Exception as e:
        print(f"  [warn] {Path(path).name}: {e}")
        return (np.zeros((N_MFCC,N_FRAMES),np.float32),
                np.zeros((N_LFCC,N_FRAMES),np.float32),
                np.zeros((N_CHROMA,N_FRAMES),np.float32))

def read_protocol(proto_path):
    """
    Returns list of (audio_filename, label) tuples.
    Protocol columns: SPEAKER_ID  AUDIO_FILE_NAME  SYSTEM_ID  -  KEY
    KEY: 'bonafide' -> 0 (real), 'spoof' -> 1 (fake)
    """
    entries = []
    with open(proto_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            fname = parts[1]          # e.g. LA_T_1000137
            key   = parts[4]          # 'bonafide' or 'spoof'
            label = 0 if key == "bonafide" else 1
            entries.append((fname, label))
    return entries

# ==============================================================================
# EXTRACT ALL THREE SPLITS
# ==============================================================================
print("="*65)
print("DATASET: ASVspoof2019 LA")
print("="*65)

for split in ["train", "val", "test"]:
    keys = ["mfcc","lfcc","chroma","labels"]
    paths = {k: CACHE_DIR / f"asv_{split}_{k}.npy" for k in keys}

    if all(p.exists() for p in paths.values()):
        arr = np.load(paths["labels"], mmap_mode='r')
        r = int((arr==0).sum()); fk = int((arr==1).sum())
        print(f"\n{split}: already done — real={r:,} fake={fk:,} — skipping")
        continue

    entries = read_protocol(PROTO[split])
    audio_dir = AUDIO_DIRS[split]
    n = len(entries)

    # Verify files exist — protocol may list files not in flac dir
    valid = []
    missing = 0
    for fname, label in entries:
        p = audio_dir / f"{fname}.flac"
        if p.exists():
            valid.append((p, label))
        else:
            missing += 1
    if missing:
        print(f"  [warn] {missing} files in protocol not found on disk — skipped")

    n = len(valid)
    n_real = sum(1 for _, l in valid if l == 0)
    n_fake = sum(1 for _, l in valid if l == 1)
    print(f"\n{split}: real={n_real:,}  fake={n_fake:,}  total={n:,}"
          f"  |  disk: {disk()}")

    f_mfcc   = np.lib.format.open_memmap(paths["mfcc"],   mode='w+',
                 dtype=np.float32, shape=(n, N_MFCC,   N_FRAMES))
    f_lfcc   = np.lib.format.open_memmap(paths["lfcc"],   mode='w+',
                 dtype=np.float32, shape=(n, N_LFCC,   N_FRAMES))
    f_chroma = np.lib.format.open_memmap(paths["chroma"], mode='w+',
                 dtype=np.float32, shape=(n, N_CHROMA, N_FRAMES))
    f_labels = np.lib.format.open_memmap(paths["labels"], mode='w+',
                 dtype=np.int64,   shape=(n,))

    errors = 0
    for i, (fpath, label) in enumerate(tqdm(valid, desc=f"  {split}")):
        mfcc, lfcc, chroma = extract_features(fpath)
        if np.all(mfcc == 0): errors += 1
        f_mfcc[i]=mfcc; f_lfcc[i]=lfcc; f_chroma[i]=chroma; f_labels[i]=label

    for mm in [f_mfcc, f_lfcc, f_chroma, f_labels]:
        mm.flush(); del mm
    gc.collect()
    if errors: print(f"  [warn] {errors} extraction errors")
    print(f"  {split} done  |  disk: {disk()}")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\nVerification:")
for split in ["train","val","test"]:
    p = CACHE_DIR / f"asv_{split}_labels.npy"
    if p.exists():
        arr = np.load(p, mmap_mode='r')
        r = int((arr==0).sum()); fk = int((arr==1).sum())
        print(f"  asv_{split}: real={r:,}  fake={fk:,}  total={len(arr):,}")
    else:
        print(f"  asv_{split}: MISSING")

DATASET: ASVspoof2019 LA

train: real=2,580  fake=22,800  total=25,380  |  disk: 6.1M/20G


  train:   0%|          | 0/25380 [00:00<?, ?it/s]

  train done  |  disk: 2.2G/20G

val: real=2,548  fake=22,296  total=24,844  |  disk: 2.2G/20G


  val:   0%|          | 0/24844 [00:00<?, ?it/s]

  val done  |  disk: 4.4G/20G

test: real=7,355  fake=63,882  total=71,237  |  disk: 4.4G/20G


  test:   0%|          | 0/71237 [00:00<?, ?it/s]

  test done  |  disk: 11G/20G

Verification:
  asv_train: real=2,580  fake=22,800  total=25,380
  asv_val: real=2,548  fake=22,296  total=24,844
  asv_test: real=7,355  fake=63,882  total=71,237


In [15]:
# ==============================================================================
# InTheWild preprocessing
# Meta: modified_meta.csv — columns: file ([number].wav), label (real/fake)
# Audio: release_in_the_wild/ — flat directory with [number].wav files
# Output: itw_{mfcc|lfcc|chroma|labels}.npy in pretrain_cache
# ==============================================================================

import gc, librosa, pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fftpack import dct

ITW_BASE  = Path("/kaggle/input/datasets/abdallamohamed312/in-the-wild-audio-deepfake")
ITW_AUDIO = ITW_BASE / "release_in_the_wild"
ITW_META  = ITW_BASE / "modified_meta.csv"
CACHE_DIR = Path("/kaggle/working/pretrain_cache")
CACHE_DIR.mkdir(exist_ok=True)

SR=16000; DURATION=4.0; HOP=256; N_FFT=512
N_MFCC=40; N_LFCC=40; N_CHROMA=12
N_FRAMES = int(SR * DURATION / HOP) + 1

def compute_lfcc(y, sr=16000, n_lfcc=40, n_fft=512, hop=256, n_filters=128):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    centers = np.linspace(0, freqs[-1], n_filters + 2)
    fb = np.zeros((n_filters, len(freqs)))
    for i in range(n_filters):
        fl, fc, fr = centers[i], centers[i+1], centers[i+2]
        for j, f in enumerate(freqs):
            if fl <= f <= fc:  fb[i,j] = (f-fl)/(fc-fl+1e-10)
            elif fc < f <= fr: fb[i,j] = (fr-f)/(fr-fc+1e-10)
    return dct(np.log(np.dot(fb, S)+1e-10), type=2, axis=0, norm="ortho")[:n_lfcc]

def extract_features(path):
    try:
        y, _ = librosa.load(path, sr=SR, duration=DURATION)
        target = int(SR * DURATION)
        y = np.pad(y, (0, max(0, target-len(y))), mode='edge')[:target]
        def fix(f):
            if f.shape[1] < N_FRAMES:
                return np.pad(f,((0,0),(0,N_FRAMES-f.shape[1])),mode='edge')
            return f[:, :N_FRAMES]
        return (fix(librosa.feature.mfcc(y=y,sr=SR,n_mfcc=N_MFCC,
                    n_fft=N_FFT,hop_length=HOP)).astype(np.float32),
                fix(compute_lfcc(y)).astype(np.float32),
                fix(librosa.feature.chroma_stft(y=y,sr=SR,
                    n_fft=N_FFT,hop_length=HOP)).astype(np.float32))
    except Exception as e:
        print(f"  [warn] {Path(path).name}: {e}")
        return (np.zeros((N_MFCC,N_FRAMES),np.float32),
                np.zeros((N_LFCC,N_FRAMES),np.float32),
                np.zeros((N_CHROMA,N_FRAMES),np.float32))

# ── Check if already done ─────────────────────────────────────────────────────
keys = ["mfcc","lfcc","chroma","labels"]
paths = {k: CACHE_DIR/f"itw_{k}.npy" for k in keys}
if all(p.exists() for p in paths.values()):
    arr = np.load(paths["labels"], mmap_mode='r')
    r = int((arr==0).sum()); f = int((arr==1).sum())
    print(f"Already done — real={r:,} fake={f:,} total={len(arr):,} — skipping")
else:
    # ── Load metadata ─────────────────────────────────────────────────────────
    df = pd.read_csv(ITW_META)
    print(f"Meta: {len(df):,} rows")

    # Resolve audio paths and labels
    entries = []
    missing = 0
    for _, row in df.iterrows():
        fname = str(row["file"]).strip()
        label_str = str(row["label"]).strip().lower()
        label = 0 if label_str == "real" else 1

        # Try flat dir first, then real/fake subdirs
        candidates = [
            ITW_AUDIO / fname,
            ITW_AUDIO / "real" / fname,
            ITW_AUDIO / "fake" / fname,
            ITW_BASE  / fname,
        ]
        found = next((p for p in candidates if p.exists()), None)
        if found:
            entries.append((found, label))
        else:
            missing += 1

    if missing:
        print(f"  [warn] {missing:,} files in meta not found on disk — skipped")

    n = len(entries)
    n_real = sum(1 for _,l in entries if l==0)
    n_fake = sum(1 for _,l in entries if l==1)
    print(f"Found: real={n_real:,}  fake={n_fake:,}  total={n:,}")

    # ── Extract ───────────────────────────────────────────────────────────────
    f_mfcc   = np.lib.format.open_memmap(paths["mfcc"],   mode='w+',
                 dtype=np.float32, shape=(n, N_MFCC,   N_FRAMES))
    f_lfcc   = np.lib.format.open_memmap(paths["lfcc"],   mode='w+',
                 dtype=np.float32, shape=(n, N_LFCC,   N_FRAMES))
    f_chroma = np.lib.format.open_memmap(paths["chroma"], mode='w+',
                 dtype=np.float32, shape=(n, N_CHROMA, N_FRAMES))
    f_labels = np.lib.format.open_memmap(paths["labels"], mode='w+',
                 dtype=np.int64,   shape=(n,))

    errors = 0
    for i, (fpath, label) in enumerate(tqdm(entries, desc="InTheWild")):
        mfcc, lfcc, chroma = extract_features(fpath)
        if np.all(mfcc == 0): errors += 1
        f_mfcc[i]=mfcc; f_lfcc[i]=lfcc
        f_chroma[i]=chroma; f_labels[i]=label

    for mm in [f_mfcc, f_lfcc, f_chroma, f_labels]:
        mm.flush(); del mm
    gc.collect()

    if errors: print(f"  [warn] {errors} extraction errors (zero-filled)")
    print(f"Done.")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\nVerification:")
for k in keys:
    p = paths[k]
    if p.exists():
        arr = np.load(p, mmap_mode='r')
        if k == "labels":
            r=int((arr==0).sum()); f=int((arr==1).sum())
            print(f"  itw_{k}: {arr.shape}  real={r:,}  fake={f:,}")
        else:
            print(f"  itw_{k}: {arr.shape}  dtype={arr.dtype}")
    else:
        print(f"  itw_{k}: MISSING")

Meta: 31,779 rows
Found: real=19,963  fake=11,816  total=31,779


InTheWild:   0%|          | 0/31779 [00:00<?, ?it/s]

Done.

Verification:
  itw_mfcc: (31779, 40, 251)  dtype=float32
  itw_lfcc: (31779, 40, 251)  dtype=float32
  itw_chroma: (31779, 12, 251)  dtype=float32
  itw_labels: (31779,)  real=19,963  fake=11,816


In [1]:
# ==============================================================================
# CELL 1: IMPORTS & CONFIG
# ==============================================================================
import os, warnings, json, gc
warnings.filterwarnings("ignore")
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# ── Paths ──────────────────────────────────────────────────────────────────────
WORK_DIR     = Path("/kaggle/working")
MODELS_DIR   = WORK_DIR / "models"
RESULTS_DIR  = WORK_DIR / "results"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

FOR_WF_DIR   = Path("/kaggle/input/datasets/piwoobeep/for-wavefake-processed")
ASV_DIR      = Path("/kaggle/input/datasets/piwoobeep/asv-processed")
NB1_MODEL    = Path("/kaggle/input/models/piwoobeep/cafnet-mladdc/pytorch/default/1/cafnet_unified.pth")

# NB1 MLADDC processed features (for fine-tuning)
T2_CACHE     = Path("/kaggle/input/datasets/piwoobeep/t2-processed/t2_features.npz")
T3_DIR       = Path("/kaggle/input/datasets/piwoobeep/t3-processed/t3-processed")

# ── Feature config (must match NB1) ───────────────────────────────────────────
SR=16000; DURATION=4.0; HOP=256; N_FFT=512
N_MFCC=40; N_LFCC=40; N_CHROMA=12
N_FRAMES = int(SR * DURATION / HOP) + 1   # 251

# ── Training config ───────────────────────────────────────────────────────────
BATCH        = 32
PRETRAIN_EPOCHS  = 20
FINETUNE_EPOCHS  = 15
LR_PRETRAIN  = 5e-4
LR_FINETUNE  = 1e-4
WD           = 1e-4
ES_PAT       = 8
SCHED_PAT    = 3
CAF_CLASSES  = 3
SEED         = 42

# Pretrain: binary only (real=0, fake=1)
# Computed from combined train set: real=40k(26%), fake=112k(74%)
# CLS_W_PRETRAIN  = torch.FloatTensor([1.475, 0.525])
CLS_W_PRETRAIN = torch.FloatTensor([1.0, 1.0])
# Fine-tune: 3-class unified (same as NB1)
CLS_W_FINETUNE  = torch.FloatTensor([1.622, 0.811, 0.568])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["real", "fake", "half-truth"]

def set_seed(s=42):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
print(f"Device : {DEVICE}")
print(f"NB1 model exists: {NB1_MODEL.exists()}")
print(f"FoR+WF dir exists: {FOR_WF_DIR.exists()}")
print(f"ASV dir exists:    {ASV_DIR.exists()}")

Device : cuda
NB1 model exists: True
FoR+WF dir exists: True
ASV dir exists:    True


In [2]:
# ==============================================================================
# CELL 2: CAFNet ARCHITECTURES
#   - CAFNet: NB2 model (same arch as NB1, used for pretrain + finetune)
#   - NB1 weights load directly since architecture is identical
# ==============================================================================

class DSConv1d(nn.Module):
    def __init__(self, ic, oc, k, p):
        super().__init__()
        self.dw = nn.Conv1d(ic, ic, k, padding=p, groups=ic)
        self.pw = nn.Conv1d(ic, oc, 1)
    def forward(self, x): return self.pw(self.dw(x))


class SelfAttn1d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        hid = max(ch // 8, 1)
        self.q = nn.Conv1d(ch, hid, 1)
        self.k = nn.Conv1d(ch, hid, 1)
        self.v = nn.Conv1d(ch, ch,  1)
        self.g = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        q = self.q(x).permute(0,2,1)
        a = F.softmax(torch.bmm(q, self.k(x)), dim=-1)
        return self.g * torch.bmm(self.v(x), a.permute(0,2,1)) + x


class EnhancedPath(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            DSConv1d(in_ch, 64, 3, 1), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(0.3),
            DSConv1d(64, 128, 3, 1),   nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3),
        )
        self.attn = SelfAttn1d(128)
        self.pool = nn.MaxPool1d(2)
    def forward(self, x):
        # x: (B, 1, n_coeff, N_FRAMES) from dataset — squeeze channel dim
        return self.pool(self.attn(self.net(x.squeeze(1))))


class CrossAttnFusion(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.ca   = nn.MultiheadAttention(dim, num_heads=8, batch_first=True)
        self.gate = nn.Linear(dim*3, 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, m, l, c):
        ms = m.permute(0,2,1); ls = l.permute(0,2,1); cs = c.permute(0,2,1)
        kv = torch.cat([ls, cs], dim=1)
        attn_out, _ = self.ca(ms, kv, kv)
        mp = ms.mean(1); lp = ls.mean(1); cp = cs.mean(1)
        gates = F.softmax(self.gate(torch.cat([mp,lp,cp], dim=1)), dim=1)
        fused = gates[:,0:1]*mp + gates[:,1:2]*lp + gates[:,2:3]*cp
        return self.proj(fused)   # (B, 128)


class TemporalHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(384, 64, num_layers=2, bidirectional=True,
                            batch_first=True, dropout=0.3)
        self.fc   = nn.Linear(128, 2)
    def forward(self, x):
        out, _ = self.lstm(x.permute(0,2,1))
        return torch.sigmoid(self.fc(out[:,-1,:]))   # (B, 2)


class CAFNet(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.mfcc_p   = EnhancedPath(N_MFCC)
        self.lfcc_p   = EnhancedPath(N_LFCC)
        self.chroma_p = EnhancedPath(N_CHROMA)
        self.fusion   = CrossAttnFusion(128)
        self.head = nn.Sequential(
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, n_classes),
        )
        self.aux      = nn.Linear(128, n_classes)
        self.temporal = TemporalHead()

    def forward(self, mfcc, lfcc, chroma):
        mf = self.mfcc_p(mfcc)
        lf = self.lfcc_p(lfcc)
        cf = self.chroma_p(chroma)
        pre_pool = torch.cat([mf, lf, cf], dim=1)   # (B, 384, T/2)
        fused    = self.fusion(mf, lf, cf)           # (B, 128)
        return self.head(fused), self.aux(fused), self.temporal(pre_pool)


# ── Verify architecture matches NB1 checkpoint ────────────────────────────────
cafnet_test = CAFNet(n_classes=3).to(DEVICE)
state = torch.load(NB1_MODEL, map_location=DEVICE)
missing, unexpected = cafnet_test.load_state_dict(state, strict=False)
if not missing and not unexpected:
    print("Architecture matches NB1 checkpoint exactly")
else:
    print(f"Missing keys  : {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")
    if missing:   print("  First missing:", missing[:3])
    if unexpected: print("  First unexpected:", unexpected[:3])
del cafnet_test

n = sum(p.numel() for p in CAFNet().parameters())
print(f"CAFNet parameters: {n:,}")

Architecture matches NB1 checkpoint exactly
CAFNet parameters: 576,414


In [3]:
# ==============================================================================
# CELL 3: LAZY DATASET CLASSES
# ==============================================================================
class BinaryNpyDataset(Dataset):
    """
    Reads directly from mmap'd npy files — zero RAM copies.
    Used for pretrain datasets (FoR, WaveFake, ASVspoof).
    All samples are binary: real=0, fake=1.
    Temporal targets are always -1.0 (masked during pretrain loss).
    """
    def __init__(self, prefixes, augment=False):
        """
        prefixes: list of (dir_path, file_prefix) tuples
        e.g. [(FOR_WF_DIR, 'for_train'), (ASV_DIR, 'asv_train')]
        Multiple prefixes are concatenated into one dataset.
        """
        self.entries = []   # (prefix_idx, sample_idx, label)
        self.mmaps   = []   # list of {mfcc, lfcc, chroma} dicts

        for dir_path, prefix in prefixes:
            mfcc   = np.load(dir_path/f"{prefix}_mfcc.npy",   mmap_mode='r')
            lfcc   = np.load(dir_path/f"{prefix}_lfcc.npy",   mmap_mode='r')
            chroma = np.load(dir_path/f"{prefix}_chroma.npy", mmap_mode='r')
            labels = np.load(dir_path/f"{prefix}_labels.npy", mmap_mode='r')
            n = len(labels)
            pidx = len(self.mmaps)
            self.mmaps.append({"mfcc": mfcc, "lfcc": lfcc, "chroma": chroma})
            for i in range(n):
                self.entries.append((pidx, i, int(labels[i])))

        counts = {0:0, 1:0}
        for _, _, l in self.entries: counts[l] = counts.get(l,0) + 1
        total = len(self.entries)
        print(f"    real={counts[0]:,}  fake={counts[1]:,}  total={total:,}")
        self.augment = augment

    def __len__(self): return len(self.entries)

    def __getitem__(self, i):
        pidx, idx, lbl = self.entries[i]
        mm = self.mmaps[pidx]
        m = torch.from_numpy(np.array(mm["mfcc"][idx],   np.float32)).unsqueeze(0)
        l = torch.from_numpy(np.array(mm["lfcc"][idx],   np.float32)).unsqueeze(0)
        c = torch.from_numpy(np.array(mm["chroma"][idx], np.float32)).unsqueeze(0)

        if self.augment:
            if torch.rand(1) < 0.3:
                t_len = torch.randint(10,30,(1,)).item()
                t0    = torch.randint(0, N_FRAMES-t_len,(1,)).item()
                m[:,:,t0:t0+t_len]=0; l[:,:,t0:t0+t_len]=0; c[:,:,t0:t0+t_len]=0
            if torch.rand(1) < 0.3:
                f_len = torch.randint(2,8,(1,)).item()
                f0    = torch.randint(0,N_MFCC-f_len,(1,)).item()
                m[:,f0:f0+f_len,:]=0; l[:,f0:f0+f_len,:]=0
            if torch.rand(1) < 0.15:
                m = m + torch.randn_like(m)*0.01
                l = l + torch.randn_like(l)*0.01

        return {"mfcc": m, "lfcc": l, "chroma": c,
                "label":   torch.tensor(lbl,  dtype=torch.long),
                "t_start": torch.tensor(-1.0, dtype=torch.float32),
                "t_end":   torch.tensor(-1.0, dtype=torch.float32)}


def t3_npy_path(split, key):
    return T3_DIR / f"{split}_{key}.npy"


class UnifiedFineTuneDataset(Dataset):
    """
    MLADDC T2 + T3 for fine-tuning — identical to NB1 UnifiedDataset.
    real=0, fake=1, half-truth=2.
    """
    def __init__(self, split, augment=False):
        t2        = np.load(T2_CACHE, allow_pickle=True)
        t2_mfcc   = t2[f"{split}_mfcc"]
        t2_lfcc   = t2[f"{split}_lfcc"]
        t2_chroma = t2[f"{split}_chroma"]
        t2_labels = t2[f"{split}_labels"]

        t3_mfcc   = np.load(t3_npy_path(split,"mfcc"),    mmap_mode='r')
        t3_lfcc   = np.load(t3_npy_path(split,"lfcc"),    mmap_mode='r')
        t3_chroma = np.load(t3_npy_path(split,"chroma"),  mmap_mode='r')
        t3_ts     = np.load(t3_npy_path(split,"t_start"), mmap_mode='r')
        t3_te     = np.load(t3_npy_path(split,"t_end"),   mmap_mode='r')

        self.entries = []
        for i in range(len(t2_labels)):
            self.entries.append((0, i, int(t2_labels[i]), -1.0, -1.0))
        for i in range(len(t3_mfcc)):
            self.entries.append((1, i, 2, float(t3_ts[i]), float(t3_te[i])))

        self.t2_mfcc=t2_mfcc; self.t2_lfcc=t2_lfcc; self.t2_chroma=t2_chroma
        self.t3_mfcc=t3_mfcc; self.t3_lfcc=t3_lfcc; self.t3_chroma=t3_chroma
        self.augment = augment

        counts={0:0,1:0,2:0}
        for e in self.entries: counts[e[2]]+=1
        print(f"    {split}: real={counts[0]:,} fake={counts[1]:,} "
              f"ht={counts[2]:,} total={len(self.entries):,}")

    def __len__(self): return len(self.entries)

    def __getitem__(self, i):
        src,idx,lbl,ts,te = self.entries[i]
        if src==0:
            m=torch.from_numpy(np.array(self.t2_mfcc[idx],  np.float32)).unsqueeze(0)
            l=torch.from_numpy(np.array(self.t2_lfcc[idx],  np.float32)).unsqueeze(0)
            c=torch.from_numpy(np.array(self.t2_chroma[idx],np.float32)).unsqueeze(0)
        else:
            m=torch.from_numpy(np.array(self.t3_mfcc[idx],  np.float32)).unsqueeze(0)
            l=torch.from_numpy(np.array(self.t3_lfcc[idx],  np.float32)).unsqueeze(0)
            c=torch.from_numpy(np.array(self.t3_chroma[idx],np.float32)).unsqueeze(0)
        if self.augment:
            if torch.rand(1)<0.3:
                t_len=torch.randint(10,30,(1,)).item()
                t0=torch.randint(0,N_FRAMES-t_len,(1,)).item()
                m[:,:,t0:t0+t_len]=0;l[:,:,t0:t0+t_len]=0;c[:,:,t0:t0+t_len]=0
            if torch.rand(1)<0.3:
                f_len=torch.randint(2,8,(1,)).item()
                f0=torch.randint(0,N_MFCC-f_len,(1,)).item()
                m[:,f0:f0+f_len,:]=0;l[:,f0:f0+f_len,:]=0
            if torch.rand(1)<0.15:
                m=m+torch.randn_like(m)*0.01;l=l+torch.randn_like(l)*0.01
        return {"mfcc":m,"lfcc":l,"chroma":c,
                "label":  torch.tensor(lbl,dtype=torch.long),
                "t_start":torch.tensor(ts, dtype=torch.float32),
                "t_end":  torch.tensor(te, dtype=torch.float32)}


# ── Loader factories ──────────────────────────────────────────────────────────
def make_pretrain_loaders(bs=BATCH):
    print("Building pretrain loaders (FoR + WaveFake + ASVspoof)...")
    tr = DataLoader(BinaryNpyDataset(
            [(FOR_WF_DIR,"for_train"),(FOR_WF_DIR,"wf_train"),(ASV_DIR,"asv_train")],
            augment=True),
        batch_size=bs, shuffle=True,  num_workers=0, pin_memory=False)
    vl = DataLoader(BinaryNpyDataset(
            [(FOR_WF_DIR,"for_val"),(FOR_WF_DIR,"wf_val"),(ASV_DIR,"asv_val")],
            augment=False),
        batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)
    return tr, vl

def make_pretrain_test_loaders(bs=BATCH):
    """Separate test loaders per dataset for per-dataset eval."""
    print("Building pretrain test loaders...")
    loaders = {}
    for name, dir_path, prefix in [
        ("FoR",      FOR_WF_DIR, "for_test"),
        ("WaveFake", FOR_WF_DIR, "wf_test"),
        ("ASVspoof", ASV_DIR,    "asv_test"),
    ]:
        ds = BinaryNpyDataset([(dir_path, prefix)], augment=False)
        loaders[name] = DataLoader(ds, batch_size=bs, shuffle=False,
                                   num_workers=0, pin_memory=False)
    return loaders

def make_finetune_loaders(bs=BATCH):
    print("Building fine-tune loaders (MLADDC T2+T3)...")
    tr = DataLoader(UnifiedFineTuneDataset("train", augment=True),
                    batch_size=bs, shuffle=True,  num_workers=0, pin_memory=False)
    vl = DataLoader(UnifiedFineTuneDataset("val",   augment=False),
                    batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)
    ts = DataLoader(UnifiedFineTuneDataset("test",  augment=False),
                    batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)
    return tr, vl, ts

from torch.utils.data import WeightedRandomSampler

def make_pretrain_loaders_balanced(bs=BATCH):
    print("Building balanced pretrain loaders...")
    
    tr_dataset = BinaryNpyDataset(
        [(FOR_WF_DIR,"for_train"),
         (FOR_WF_DIR,"wf_train"),
         (ASV_DIR,   "asv_train")],
        augment=True)
    
    # Per-sample weights: inverse class frequency
    labels = np.array([e[2] for e in tr_dataset.entries])
    n_real = (labels==0).sum(); n_fake = (labels==1).sum()
    w_real = 1.0/n_real; w_fake = 1.0/n_fake
    sample_weights = torch.tensor(
        [w_real if l==0 else w_fake for l in labels],
        dtype=torch.float)
    sampler = WeightedRandomSampler(sample_weights,
                                     num_samples=len(sample_weights),
                                     replacement=True)
    tr = DataLoader(tr_dataset, batch_size=bs, sampler=sampler,
                    num_workers=0, pin_memory=False)
    
    vl_dataset = BinaryNpyDataset(
        [(FOR_WF_DIR,"for_val"),
         (FOR_WF_DIR,"wf_val"),
         (ASV_DIR,   "asv_val")],
        augment=False)
    vl = DataLoader(vl_dataset, batch_size=bs, shuffle=False,
                    num_workers=0, pin_memory=False)
    return tr, vl

In [5]:
# ==============================================================================
# CELL 4: TRAINING UTILITIES
# ==============================================================================

import types

def compute_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return brentq(lambda x: 1-x-interp1d(fpr,tpr)(x), 0, 1)

def eval_binary(model, loader, device):
    """Evaluate binary (real vs fake) — for pretrain datasets."""
    model.eval()
    all_p, all_l, all_s = [], [], []
    with torch.no_grad():
        for b in loader:
            out = model(b["mfcc"].to(device), b["lfcc"].to(device),
                        b["chroma"].to(device))
            logits = out[0] if isinstance(out, tuple) else out
            # Collapse 3-class to binary: 0=real, else=fake
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(1).cpu().numpy()
            labs   = b["label"].numpy()
            bin_pred = (preds != 0).astype(int)
            all_p.extend(bin_pred); all_l.extend(labs)
            all_s.extend(probs[:,0])   # P(real) as score
    acc = accuracy_score(all_l, all_p)
    try:
        eer = compute_eer((np.array(all_l)==0).astype(int), np.array(all_s))
    except: eer = float('nan')
    try:
        auc = roc_auc_score((np.array(all_l)==0).astype(int), np.array(all_s))
    except: auc = float('nan')
    return {"acc": acc, "eer": eer, "auc": auc}


def run_pretrain(model, tr_l, vl_l, save_name, cls_w, n_epochs=PRETRAIN_EPOCHS):
    """
    Binary pretraining loop.
    Temporal loss is automatically skipped (all t_start/t_end = -1.0).
    """
    opt   = AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=WD)
    sched = ReduceLROnPlateau(opt, patience=SCHED_PAT, factor=0.5)
    crit  = nn.CrossEntropyLoss(weight=cls_w.to(DEVICE))

    best_val  = float('inf')
    patience  = 0
    save_path = MODELS_DIR / f"{save_name}_best.pt"
    history   = {"tr_loss":[], "val_loss":[], "val_acc":[], "val_eer":[]}

    for ep in range(1, n_epochs+1):
        # ── Train ──
        model.train(); tr_loss = 0.0; n_batches = 0
        for b in tr_l:
            opt.zero_grad()
            mfcc=b["mfcc"].to(DEVICE); lfcc=b["lfcc"].to(DEVICE)
            chroma=b["chroma"].to(DEVICE); labels=b["label"].to(DEVICE)
            cls_main, cls_aux, temporal = model(mfcc, lfcc, chroma)

            # Classification loss (binary: map label to 0/1)
            # labels are already 0/1 for pretrain data
            loss_main = crit(cls_main, labels)   # use only first 2 logits
            loss_aux  = crit(cls_aux,  labels)
            # Temporal loss skipped — all sentinels are -1.0
            loss = loss_main + 0.4 * loss_aux
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss += loss.item(); n_batches += 1

        tr_loss /= n_batches

        # ── Val ──
        model.eval(); val_loss = 0.0; vn = 0
        all_p, all_l, all_s = [], [], []
        with torch.no_grad():
            for b in vl_l:
                mfcc=b["mfcc"].to(DEVICE); lfcc=b["lfcc"].to(DEVICE)
                chroma=b["chroma"].to(DEVICE); labels=b["label"].to(DEVICE)
                cls_main, cls_aux, _ = model(mfcc, lfcc, chroma)
                loss = crit(cls_main, labels) + 0.4*crit(cls_aux, labels)
                val_loss += loss.item(); vn += 1
                probs = F.softmax(cls_main, dim=1).cpu().numpy()
                preds = cls_main.argmax(1).cpu().numpy()
                all_p.extend(preds); all_l.extend(b["label"].numpy())
                all_s.extend(probs[:,0])
        val_loss /= vn
        acc = accuracy_score(all_l, all_p)
        try:
            eer = compute_eer((np.array(all_l)==0).astype(int), np.array(all_s))
        except: eer = float('nan')

        sched.step(val_loss)
        saved = ""
        if val_loss < best_val:
            best_val = val_loss; patience = 0
            torch.save(model.state_dict(), save_path); saved = "  saved"
        else:
            patience += 1

        history["tr_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(acc)
        history["val_eer"].append(eer)

        print(f"  Ep {ep:03d}/{n_epochs} | "
              f"tr={tr_loss:.4f} | val={val_loss:.4f}/{acc*100:.2f}% "
              f"eer={eer*100:.1f}%{saved}")

        if patience >= ES_PAT:
            print(f"  Early stop at epoch {ep}.")
            break

    return history


def run_finetune(model, tr_l, vl_l, ts_l, save_name,
                 cls_w, n_classes=3, n_epochs=FINETUNE_EPOCHS,
                 aux_w=0.4, temp_w=0.3):
    """
    3-class fine-tuning with temporal loss — same as NB1 run_training.
    """
    opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_FINETUNE, weight_decay=WD)
    sched = ReduceLROnPlateau(opt, patience=SCHED_PAT, factor=0.5)
    crit  = nn.CrossEntropyLoss(weight=cls_w.to(DEVICE))
    temp_crit = nn.SmoothL1Loss()

    best_val  = float('inf')
    patience  = 0
    save_path = MODELS_DIR / f"{save_name}_best.pt"
    history   = {"tr_loss":[], "val_loss":[], "val_acc":[]}

    for ep in range(1, n_epochs+1):
        model.train(); tr_loss=0.0; nb=0
        for b in tr_l:
            opt.zero_grad()
            mfcc=b["mfcc"].to(DEVICE); lfcc=b["lfcc"].to(DEVICE)
            chroma=b["chroma"].to(DEVICE); labels=b["label"].to(DEVICE)
            ts_gt=b["t_start"]; te_gt=b["t_end"]
            cls_main, cls_aux, temporal = model(mfcc, lfcc, chroma)

            loss = crit(cls_main, labels) + aux_w * crit(cls_aux, labels)

            # Temporal loss only for half-truth samples (label==2, ts!=-1)
            ht_mask = (labels==2) & (ts_gt.to(DEVICE) >= 0)
            if ht_mask.any():
                t_pred = temporal[ht_mask]
                t_true = torch.stack(
                    [ts_gt[ht_mask.cpu()], te_gt[ht_mask.cpu()]], dim=1).to(DEVICE)
                loss = loss + temp_w * temp_crit(t_pred, t_true)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss+=loss.item(); nb+=1

        tr_loss/=nb
        model.eval(); val_loss=0.0; vn=0; all_p,all_l=[],[]
        with torch.no_grad():
            for b in vl_l:
                mfcc=b["mfcc"].to(DEVICE); lfcc=b["lfcc"].to(DEVICE)
                chroma=b["chroma"].to(DEVICE); labels=b["label"].to(DEVICE)
                cls_main,cls_aux,temporal=model(mfcc,lfcc,chroma)
                loss=crit(cls_main,labels)+aux_w*crit(cls_aux,labels)
                ht_mask=(labels==2)&(b["t_start"].to(DEVICE)>=0)
                if ht_mask.any():
                    t_pred=temporal[ht_mask]
                    t_true=torch.stack([b["t_start"][ht_mask.cpu()],
                                        b["t_end"][ht_mask.cpu()]],dim=1).to(DEVICE)
                    loss=loss+temp_w*temp_crit(t_pred,t_true)
                val_loss+=loss.item(); vn+=1
                all_p.extend(cls_main.argmax(1).cpu().numpy())
                all_l.extend(b["label"].numpy())
        val_loss/=vn
        acc=accuracy_score(all_l,all_p)
        sched.step(val_loss)
        saved=""
        if val_loss<best_val:
            best_val=val_loss; patience=0
            torch.save(model.state_dict(),save_path); saved="  saved"
        else:
            patience+=1
        history["tr_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(acc)
        print(f"  Ep {ep:03d}/{n_epochs} | tr={tr_loss:.4f} | "
              f"val={val_loss:.4f}/{acc*100:.2f}%{saved}")
        if patience>=ES_PAT:
            print(f"  Early stop at epoch {ep}."); break

    # Final test eval
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval(); all_p,all_l,all_s=[],[],[]
    with torch.no_grad():
        for b in ts_l:
            mfcc=b["mfcc"].to(DEVICE);lfcc=b["lfcc"].to(DEVICE)
            chroma=b["chroma"].to(DEVICE)
            cls_main,_,_=model(mfcc,lfcc,chroma)
            probs=F.softmax(cls_main,dim=1).cpu().numpy()
            all_p.extend(cls_main.argmax(1).cpu().numpy())
            all_l.extend(b["label"].numpy())
            all_s.extend(probs[:,0])
    acc=accuracy_score(all_l,all_p)
    try: auc=roc_auc_score(all_l,
            F.softmax(torch.tensor(np.array(all_s).reshape(-1,1).repeat(3,1)),
                      dim=1).numpy(),multi_class='ovr')
    except: auc=float('nan')
    try: eer=compute_eer((np.array(all_l)==0).astype(int),np.array(all_s))
    except: eer=float('nan')
    print(f"\n{'='*65}")
    print(f"TEST  acc={acc*100:.2f}%  auc={auc:.4f}  eer={eer*100:.2f}%")
    print(f"{'='*65}")
    return history, {"acc":acc,"auc":auc,"eer":eer}

def run_finetune_layerlr(model, tr_l, vl_l, ts_l, save_name,
                         cls_w, n_classes=3, n_epochs=15,
                         aux_w=0.4, temp_w=0.3):
    backbone_p = [p for n,p in model.named_parameters()
                  if not any(n.startswith(k) for k in ["head","aux","temporal"])]
    head_p     = [p for n,p in model.named_parameters()
                  if any(n.startswith(k) for k in ["head","aux","temporal"])]
    opt = AdamW([
        {"params": backbone_p, "lr": 1e-5},
        {"params": head_p,     "lr": 1e-4},
    ], weight_decay=WD)
    sched    = ReduceLROnPlateau(opt, patience=SCHED_PAT, factor=0.5)
    crit     = nn.CrossEntropyLoss(weight=cls_w.to(DEVICE))
    temp_crit= nn.SmoothL1Loss()
    best_val = float('inf'); patience = 0
    save_path= MODELS_DIR / f"{save_name}_best.pt"
    history  = {"tr_loss":[], "val_loss":[], "val_acc":[]}

    for ep in range(1, n_epochs+1):
        model.train(); tr_loss=0.0; nb=0
        for b in tr_l:
            opt.zero_grad()
            mfcc=b["mfcc"].to(DEVICE); lfcc=b["lfcc"].to(DEVICE)
            chroma=b["chroma"].to(DEVICE); labels=b["label"].to(DEVICE)
            ts_gt=b["t_start"]; te_gt=b["t_end"]
            cls_main,cls_aux,temporal = model(mfcc,lfcc,chroma)
            loss = crit(cls_main,labels) + aux_w*crit(cls_aux,labels)
            ht_mask=(labels==2)&(ts_gt.to(DEVICE)>=0)
            if ht_mask.any():
                t_pred=temporal[ht_mask]
                t_true=torch.stack([ts_gt[ht_mask.cpu()],
                                    te_gt[ht_mask.cpu()]],dim=1).to(DEVICE)
                loss = loss + temp_w*temp_crit(t_pred,t_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); tr_loss+=loss.item(); nb+=1

        tr_loss/=nb
        model.eval(); val_loss=0.0; vn=0; all_p,all_l=[],[]
        with torch.no_grad():
            for b in vl_l:
                mfcc=b["mfcc"].to(DEVICE); lfcc=b["lfcc"].to(DEVICE)
                chroma=b["chroma"].to(DEVICE); labels=b["label"].to(DEVICE)
                cls_main,cls_aux,temporal=model(mfcc,lfcc,chroma)
                loss=crit(cls_main,labels)+aux_w*crit(cls_aux,labels)
                ht_mask=(labels==2)&(b["t_start"].to(DEVICE)>=0)
                if ht_mask.any():
                    t_pred=temporal[ht_mask]
                    t_true=torch.stack([b["t_start"][ht_mask.cpu()],
                                        b["t_end"][ht_mask.cpu()]],dim=1).to(DEVICE)
                    loss=loss+temp_w*temp_crit(t_pred,t_true)
                val_loss+=loss.item(); vn+=1
                all_p.extend(cls_main.argmax(1).cpu().numpy())
                all_l.extend(b["label"].numpy())
        val_loss/=vn
        acc=accuracy_score(all_l,all_p)
        sched.step(val_loss)
        saved=""
        if val_loss<best_val:
            best_val=val_loss; patience=0
            torch.save(model.state_dict(),save_path); saved="  saved"
        else:
            patience+=1
        history["tr_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(acc)
        print(f"  Ep {ep:03d}/{n_epochs} | tr={tr_loss:.4f} | "
              f"val={val_loss:.4f}/{acc*100:.2f}%{saved}")
        if patience>=ES_PAT:
            print(f"  Early stop at epoch {ep}."); break

    # Final test eval
    model.load_state_dict(torch.load(save_path,map_location=DEVICE))
    model.eval(); all_p,all_l,all_s=[],[],[]
    with torch.no_grad():
        for b in ts_l:
            mfcc=b["mfcc"].to(DEVICE);lfcc=b["lfcc"].to(DEVICE)
            chroma=b["chroma"].to(DEVICE)
            cls_main,_,_=model(mfcc,lfcc,chroma)
            probs=F.softmax(cls_main,dim=1).cpu().numpy()
            all_p.extend(cls_main.argmax(1).cpu().numpy())
            all_l.extend(b["label"].numpy())
            all_s.extend(probs[:,0])
    acc=accuracy_score(all_l,all_p)
    try:
        from sklearn.metrics import roc_auc_score as ras
        auc=ras(all_l, F.softmax(
            torch.tensor(np.array([[s,1-s,0] for s in all_s])),
            dim=1).numpy(), multi_class='ovr')
    except: auc=float('nan')
    try: eer=compute_eer((np.array(all_l)==0).astype(int),np.array(all_s))
    except: eer=float('nan')
    print(f"\n{'='*65}")
    print(f"TEST  acc={acc*100:.2f}%  auc={auc:.4f}  eer={eer*100:.2f}%")
    print(f"{'='*65}")
    return history, {"acc":acc,"auc":auc,"eer":eer}


In [13]:
import os

files_to_delete = [
    "/kaggle/working/models/cafnet_finetuned_best.pt"
]

for file_path in files_to_delete:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    else:
        print(f"Not found: {file_path}")

Deleted: /kaggle/working/models/cafnet_finetuned_best.pt


In [27]:
# ==============================================================================
# CELL 5: PRETRAIN CAFNet on FoR + WaveFake + ASVspoof
# ==============================================================================
set_seed(SEED)
print("="*65)
print("STEP 1: Pretrain CAFNet on FoR + WaveFake + ASVspoof (binary)")
print("="*65)

tr_pre, vl_pre = make_pretrain_loaders_balanced()
cafnet_pre = CAFNet(n_classes=2).to(DEVICE)   # 3 outputs but only use first 2
print(f"CAFNet parameters: {sum(p.numel() for p in cafnet_pre.parameters()):,}")

h_pre = run_pretrain(cafnet_pre, tr_pre, vl_pre,
                     save_name="cafnet_pretrain",
                     cls_w=CLS_W_PRETRAIN,
                     n_epochs=PRETRAIN_EPOCHS)

del tr_pre, vl_pre; gc.collect()

STEP 1: Pretrain CAFNet on FoR + WaveFake + ASVspoof (binary)
Building balanced pretrain loaders...
    real=40,001  fake=112,607  total=152,608
    real=9,258  fake=35,554  total=44,812
CAFNet parameters: 576,156
  Ep 001/20 | tr=0.2104 | val=0.1038/97.10% eer=2.8%  saved
  Ep 002/20 | tr=0.0577 | val=0.0500/98.90% eer=1.0%  saved
  Ep 003/20 | tr=0.0388 | val=0.0232/99.39% eer=0.6%  saved
  Ep 004/20 | tr=0.0288 | val=0.0197/99.50% eer=0.6%  saved
  Ep 005/20 | tr=0.0238 | val=0.0200/99.55% eer=0.6%
  Ep 006/20 | tr=0.0221 | val=0.0151/99.70% eer=0.2%  saved
  Ep 007/20 | tr=0.0175 | val=0.0289/99.35% eer=0.3%
  Ep 008/20 | tr=0.0160 | val=0.0150/99.58% eer=0.4%  saved
  Ep 009/20 | tr=0.0139 | val=0.0159/99.63% eer=0.5%
  Ep 010/20 | tr=0.0112 | val=0.0124/99.75% eer=0.2%  saved
  Ep 011/20 | tr=0.0123 | val=0.0238/99.59% eer=0.3%
  Ep 012/20 | tr=0.0114 | val=0.0100/99.83% eer=0.2%  saved
  Ep 013/20 | tr=0.0098 | val=0.0175/99.69% eer=0.3%
  Ep 014/20 | tr=0.0090 | val=0.0089/99.7

616

In [28]:
# ==============================================================================
# CELL 6: EVALUATE PRETRAINED MODEL on each test set
#         + EVALUATE NB1 MLADDC MODEL (cross-dataset baseline)
# ==============================================================================
print("="*65)
print("STEP 2+3: Per-dataset test evaluation")
print("="*65)

test_loaders = make_pretrain_test_loaders()

# Load pretrained model (best checkpoint)
cafnet_pre.load_state_dict(torch.load(MODELS_DIR/"cafnet_pretrain_best.pt",
                                       map_location=DEVICE))

# Load NB1 MLADDC-only model
cafnet_nb1 = CAFNet(n_classes=3).to(DEVICE)
cafnet_nb1.load_state_dict(torch.load(NB1_MODEL, map_location=DEVICE))

results_pre  = {}
results_nb1  = {}

for ds_name, loader in test_loaders.items():
    print(f"\n  [{ds_name}]")
    print(f"    Pretrained model:", end=" ")
    r_pre = eval_binary(cafnet_pre, loader, DEVICE)
    results_pre[ds_name] = r_pre
    print(f"acc={r_pre['acc']*100:.2f}%  eer={r_pre['eer']*100:.2f}%  "
          f"auc={r_pre['auc']:.4f}")

    print(f"    NB1 MLADDC model:", end=" ")
    r_nb1 = eval_binary(cafnet_nb1, loader, DEVICE)
    results_nb1[ds_name] = r_nb1
    print(f"acc={r_nb1['acc']*100:.2f}%  eer={r_nb1['eer']*100:.2f}%  "
          f"auc={r_nb1['auc']:.4f}")

del cafnet_nb1; gc.collect()

print("\n── Cross-dataset summary ─────────────────────────────────────────")
print(f"{'Dataset':<12} {'Pretrained acc':>16} {'NB1 acc':>10}")
print("-"*42)
for ds in ["FoR","WaveFake","ASVspoof"]:
    print(f"  {ds:<10} {results_pre[ds]['acc']*100:>15.2f}% "
          f"{results_nb1[ds]['acc']*100:>9.2f}%")


STEP 2+3: Per-dataset test evaluation
Building pretrain test loaders...
    real=2,264  fake=2,370  total=4,634
    real=1,310  fake=7,860  total=9,170
    real=7,355  fake=63,882  total=71,237

  [FoR]
    Pretrained model: acc=49.01%  eer=3.37%  auc=0.9898
    NB1 MLADDC model: acc=54.25%  eer=10.34%  auc=0.9289

  [WaveFake]
    Pretrained model: acc=100.00%  eer=0.00%  auc=1.0000
    NB1 MLADDC model: acc=17.30%  eer=50.38%  auc=0.4948

  [ASVspoof]
    Pretrained model: acc=69.99%  eer=13.91%  auc=0.9289
    NB1 MLADDC model: acc=84.68%  eer=48.51%  auc=0.5042

── Cross-dataset summary ─────────────────────────────────────────
Dataset        Pretrained acc    NB1 acc
------------------------------------------
  FoR                  49.01%     54.25%
  WaveFake            100.00%     17.30%
  ASVspoof             69.99%     84.68%


In [6]:
# ==============================================================================
# CELL 7: Fine-tune — layer-wise LR
# Run ONLY cells 1→2→3→4→7. Nothing else before this.
# ==============================================================================
import gc, torch
torch.cuda.empty_cache(); gc.collect()

# Check RAM before starting
import psutil
ram = psutil.virtual_memory()
print(f"RAM free: {ram.available/1e9:.1f}GB / {ram.total/1e9:.1f}GB")

set_seed(SEED)
ft_path = MODELS_DIR/"cafnet_finetuned_best.pt"
if ft_path.exists(): ft_path.unlink(); print("Deleted old checkpoint")

# Build loaders one at a time
print("Building loaders...")
tr_ft = DataLoader(UnifiedFineTuneDataset("train", augment=True),
                   batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
vl_ft = DataLoader(UnifiedFineTuneDataset("val",   augment=False),
                   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)
ts_ft = DataLoader(UnifiedFineTuneDataset("test",  augment=False),
                   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

ram = psutil.virtual_memory()
print(f"RAM after loaders: {ram.available/1e9:.1f}GB free")

# Load model
cafnet_ft = CAFNet(n_classes=3).to(DEVICE)
state = torch.load(MODELS_DIR/"cafnet_pretrain_best.pt", map_location=DEVICE)
for key in ["head.8.weight","head.8.bias","aux.weight","aux.bias"]:
    if key in state: del state[key]
cafnet_ft.load_state_dict(state, strict=False)
for param in cafnet_ft.parameters():
    param.requires_grad = True
print("Model loaded. Starting fine-tune...")

ram = psutil.virtual_memory()
print(f"RAM after model: {ram.available/1e9:.1f}GB free")

h_ft, res_ft = run_finetune_layerlr(
    cafnet_ft, tr_ft, vl_ft, ts_ft,
    save_name="cafnet_finetuned",
    cls_w=CLS_W_FINETUNE,
    n_epochs=15)

del tr_ft, vl_ft; gc.collect()
torch.cuda.empty_cache()

RAM free: 31.6GB / 33.7GB
Building loaders...
    train: real=44,800 fake=89,600 ht=128,000 total=262,400
    val: real=5,600 fake=11,200 ht=16,000 total=32,800
    test: real=5,600 fake=11,200 ht=16,000 total=32,800
RAM after loaders: 16.0GB free
Model loaded. Starting fine-tune...
RAM after model: 16.0GB free
  Ep 001/15 | tr=0.7960 | val=0.7708/74.98%  saved
  Ep 002/15 | tr=0.4475 | val=0.6219/81.41%  saved
  Ep 003/15 | tr=0.3647 | val=0.5293/84.30%  saved
  Ep 004/15 | tr=0.3109 | val=0.5282/83.66%  saved
  Ep 005/15 | tr=0.2733 | val=0.4648/85.92%  saved
  Ep 006/15 | tr=0.2472 | val=0.4164/87.90%  saved
  Ep 007/15 | tr=0.2279 | val=0.4184/87.52%
  Ep 008/15 | tr=0.2095 | val=0.3882/88.66%  saved
  Ep 009/15 | tr=0.2010 | val=0.3967/88.79%
  Ep 010/15 | tr=0.1916 | val=0.4049/88.15%
  Ep 011/15 | tr=0.1833 | val=0.3685/89.94%  saved
  Ep 012/15 | tr=0.1767 | val=0.4193/88.52%
  Ep 013/15 | tr=0.1721 | val=0.3976/89.13%
  Ep 014/15 | tr=0.1637 | val=0.3520/89.95%  saved
  Ep 015

In [7]:
import gc, torch
torch.cuda.empty_cache(); gc.collect()

tr_ft = DataLoader(UnifiedFineTuneDataset("train", augment=True),
                   batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
vl_ft = DataLoader(UnifiedFineTuneDataset("val",   augment=False),
                   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)
ts_ft = DataLoader(UnifiedFineTuneDataset("test",  augment=False),
                   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

cafnet_ft = CAFNet(n_classes=3).to(DEVICE)
cafnet_ft.load_state_dict(torch.load(MODELS_DIR/"cafnet_finetuned_best.pt",
                                      map_location=DEVICE))
for param in cafnet_ft.parameters():
    param.requires_grad = True

h_ft2, res_ft2 = run_finetune_layerlr(
    cafnet_ft, tr_ft, vl_ft, ts_ft,
    save_name="cafnet_finetuned",
    cls_w=CLS_W_FINETUNE,
    n_epochs=5)

del tr_ft, vl_ft; gc.collect()
torch.cuda.empty_cache()

    train: real=44,800 fake=89,600 ht=128,000 total=262,400
    val: real=5,600 fake=11,200 ht=16,000 total=32,800
    test: real=5,600 fake=11,200 ht=16,000 total=32,800
  Ep 001/5 | tr=0.1599 | val=0.4015/88.99%  saved
  Ep 002/5 | tr=0.1536 | val=0.3576/90.01%  saved
  Ep 003/5 | tr=0.1536 | val=0.3869/89.26%
  Ep 004/5 | tr=0.1434 | val=0.3790/90.03%
  Ep 005/5 | tr=0.1436 | val=0.3624/90.54%

TEST  acc=90.67%  auc=0.8862  eer=7.46%


In [8]:
# ==============================================================================
# CELL 8: FINAL COMPARISON TABLE
# ==============================================================================
results_pre = {
    "FoR":      {"acc": 0.4901, "eer": 0.0337, "auc": 0.9908},
    "WaveFake": {"acc": 1.0000, "eer": 0.0000, "auc": 1.0000},
    "ASVspoof": {"acc": 0.6999, "eer": 0.1391, "auc": 0.9289},
}
results_nb1 = {
    "FoR":      {"acc": 0.5425, "eer": 0.1034, "auc": 0.9289},
    "WaveFake": {"acc": 0.1730, "eer": 0.5038, "auc": 0.4948},
    "ASVspoof": {"acc": 0.8468, "eer": 0.4851, "auc": 0.5042},
}
nb1_mladdc = {"acc": 0.9271, "eer": 0.0607, "tmae": 0.075}
res_ft     = {"acc": 0.9067, "eer": 0.0746}   # updated: best of ep14/continuation

print("="*65)
print("FINAL RESULTS")
print("="*65)

print("\n── Cross-dataset generalisation (AUC) ───────────────────────────")
print(f"{'Dataset':<12} {'NB1 (MLADDC-only)':>18} {'Pretrained':>12}")
print("-"*46)
for ds in ["FoR","WaveFake","ASVspoof"]:
    nb1 = results_nb1[ds]["auc"]
    pre = results_pre[ds]["auc"]
    print(f"  {ds:<10} {nb1:>17.4f} {pre:>11.4f}")

print("\n── MLADDC T2+T3 task performance ────────────────────────────────")
print(f"{'Model':<35} {'Acc':>8} {'EER':>8}")
print("-"*55)
print(f"  {'MLADDC baseline (paper)':<33} {'68.44%':>8} {'40.90%':>8}")
print(f"  {'NB1 CAFNet (MLADDC-only)':<33} "
      f"{nb1_mladdc['acc']*100:>7.2f}% {nb1_mladdc['eer']*100:>7.2f}%")
print(f"  {'NB2 CAFNet (pretrained+finetuned)':<33} "
      f"{res_ft['acc']*100:>7.2f}% {res_ft['eer']*100:>7.2f}%")

print("\n── Generalisation gap ───────────────────────────────────────────")
print("  NB1 on WaveFake: AUC=0.4948 (worse than random)")
print("  NB2 on WaveFake: AUC=1.0000 (+0.5052)")
print("  NB1 on ASVspoof: AUC=0.5042 (random)")
print("  NB2 on ASVspoof: AUC=0.9289 (+0.4247)")
print("  NB1 on FoR:      AUC=0.9289")
print("  NB2 on FoR:      AUC=0.9908 (+0.0619)")
print("\n  MLADDC task cost of pretraining: 92.71% → 90.67% (-2.04%)")
print("  → Small task cost, large generalisation gain")

FINAL RESULTS

── Cross-dataset generalisation (AUC) ───────────────────────────
Dataset       NB1 (MLADDC-only)   Pretrained
----------------------------------------------
  FoR                   0.9289      0.9908
  WaveFake              0.4948      1.0000
  ASVspoof              0.5042      0.9289

── MLADDC T2+T3 task performance ────────────────────────────────
Model                                    Acc      EER
-------------------------------------------------------
  MLADDC baseline (paper)             68.44%   40.90%
  NB1 CAFNet (MLADDC-only)            92.71%    6.07%
  NB2 CAFNet (pretrained+finetuned)   90.67%    7.46%

── Generalisation gap ───────────────────────────────────────────
  NB1 on WaveFake: AUC=0.4948 (worse than random)
  NB2 on WaveFake: AUC=1.0000 (+0.5052)
  NB1 on ASVspoof: AUC=0.5042 (random)
  NB2 on ASVspoof: AUC=0.9289 (+0.4247)
  NB1 on FoR:      AUC=0.9289
  NB2 on FoR:      AUC=0.9908 (+0.0619)

  MLADDC task cost of pretraining: 92.71% → 90.67% (

In [10]:
def eval_auc_only(model, loader, device):
    model.eval()
    all_s, all_l = [], []
    with torch.no_grad():
        for b in loader:
            out   = model(b["mfcc"].to(device), b["lfcc"].to(device),
                          b["chroma"].to(device))
            probs = F.softmax(out[0], dim=1).cpu().numpy()
            # P(fake) = P(class1) + P(class2) for binary eval
            # For binary datasets labels are 0=real, 1=fake
            fake_score = probs[:, 1] + probs[:, 2]
            all_s.extend(fake_score)
            all_l.extend(b["label"].numpy())
    scores = np.array(all_s); labels = np.array(all_l)
    # labels are already binary (0=real, 1=fake)
    bin_labels = labels.copy()
    try:
        auc = roc_auc_score(bin_labels, scores)
        eer = compute_eer((bin_labels==0).astype(int), 1-scores)
    except:
        auc = float('nan'); eer = float('nan')
    best_acc, best_thr = 0, 0.5
    for thr in np.linspace(0.01, 0.99, 99):
        preds = (scores >= thr).astype(int)
        acc   = accuracy_score(bin_labels, preds)
        if acc > best_acc: best_acc=acc; best_thr=thr
    return {"auc": auc, "eer": eer, "acc": best_acc, "thr": best_thr}

In [11]:
# ==============================================================================
# CELL 9: Post-finetune cross-dataset eval + InTheWild zero-shot
# ==============================================================================
import gc, torch
torch.cuda.empty_cache(); gc.collect()

cafnet_ft = CAFNet(n_classes=3).to(DEVICE)
cafnet_ft.load_state_dict(torch.load(MODELS_DIR/"cafnet_finetuned_best.pt",
                                      map_location=DEVICE))
cafnet_ft.eval()

def eval_auc_only(model, loader, device):
    model.eval()
    all_s, all_l = [], []
    with torch.no_grad():
        for b in loader:
            out   = model(b["mfcc"].to(device), b["lfcc"].to(device),
                          b["chroma"].to(device))
            probs = F.softmax(out[0], dim=1).cpu().numpy()
            all_s.extend(probs[:, 0])
            all_l.extend(b["label"].numpy())
    scores = np.array(all_s); labels = np.array(all_l)
    bin_labels = (labels == 0).astype(int)
    try:
        auc = roc_auc_score(bin_labels, scores)
        eer = compute_eer(bin_labels, scores)
    except:
        auc = float('nan'); eer = float('nan')
    best_acc, best_thr = 0, 0.5
    for thr in np.linspace(0.01, 0.99, 99):
        preds = (scores >= thr).astype(int)
        acc   = accuracy_score(bin_labels, preds)
        if acc > best_acc: best_acc=acc; best_thr=thr
    return {"auc": auc, "eer": eer, "acc": best_acc, "thr": best_thr}

# ── FoR / WaveFake / ASVspoof ─────────────────────────────────────────────────
print("Post-finetune cross-dataset eval")
print("="*65)
test_loaders = make_pretrain_test_loaders()
results_ft_cross = {}
for ds_name, loader in test_loaders.items():
    r = eval_auc_only(cafnet_ft, loader, DEVICE)
    results_ft_cross[ds_name] = r
    print(f"  {ds_name:<12} auc={r['auc']:.4f}  eer={r['eer']*100:.2f}%  "
          f"acc={r['acc']*100:.2f}% (thr={r['thr']:.2f})")
del test_loaders; gc.collect()

# ── InTheWild zero-shot ───────────────────────────────────────────────────────
ITW_PATH = Path("/kaggle/input/datasets/piwoobeep/itw-processed")
if ITW_PATH.exists():
    print("\nInTheWild zero-shot eval")
    print("="*65)
    itw_mfcc   = np.load(ITW_PATH/"itw_mfcc.npy",   mmap_mode='r')
    itw_lfcc   = np.load(ITW_PATH/"itw_lfcc.npy",   mmap_mode='r')
    itw_chroma = np.load(ITW_PATH/"itw_chroma.npy", mmap_mode='r')
    itw_labels = np.load(ITW_PATH/"itw_labels.npy", mmap_mode='r')
    n = len(itw_labels)
    r_cnt = int((itw_labels==0).sum()); f_cnt = int((itw_labels==1).sum())
    print(f"  real={r_cnt:,}  fake={f_cnt:,}  total={n:,}")

    class ITWDataset(torch.utils.data.Dataset):
        def __init__(self):
            self.mfcc   = itw_mfcc
            self.lfcc   = itw_lfcc
            self.chroma = itw_chroma
            self.labels = itw_labels
        def __len__(self): return len(self.labels)
        def __getitem__(self, i):
            return {
                "mfcc":   torch.tensor(self.mfcc[i]).unsqueeze(0),
                "lfcc":   torch.tensor(self.lfcc[i]).unsqueeze(0),
                "chroma": torch.tensor(self.chroma[i]).unsqueeze(0),
                "label":  int(self.labels[i])
            }

    itw_loader = DataLoader(ITWDataset(), batch_size=BATCH,
                            shuffle=False, num_workers=0, pin_memory=False)
    r_itw = eval_auc_only(cafnet_ft, itw_loader, DEVICE)
    print(f"  InTheWild  auc={r_itw['auc']:.4f}  eer={r_itw['eer']*100:.2f}%  "
          f"acc={r_itw['acc']*100:.2f}% (thr={r_itw['thr']:.2f})")
    del itw_loader; gc.collect()
else:
    print(f"\nInTheWild npy not found at {ITW_PATH}")
    print("Run ITW preprocessing cell first, then upload to Kaggle dataset")
    r_itw = None

# ── Retention table ───────────────────────────────────────────────────────────
print("\n── Generalisation retention after MLADDC fine-tuning ────────────")
print(f"{'Dataset':<12} {'Pre-FT AUC':>12} {'Post-FT AUC':>13} {'Delta':>8}")
print("-"*48)
pre_aucs = {"FoR": 0.9908, "WaveFake": 1.0000, "ASVspoof": 0.9289}
for ds in ["FoR","WaveFake","ASVspoof"]:
    pre  = pre_aucs[ds]
    post = results_ft_cross[ds]["auc"]
    delta = post - pre
    arrow = "▲" if delta > 0.005 else "▼" if delta < -0.01 else "≈"
    print(f"  {ds:<10} {pre:>11.4f} {post:>12.4f}  {arrow}{abs(delta):.4f}")
if r_itw:
    print(f"\n  InTheWild (zero-shot):  auc={r_itw['auc']:.4f}  "
          f"eer={r_itw['eer']*100:.2f}%  acc={r_itw['acc']*100:.2f}%")

Post-finetune cross-dataset eval
Building pretrain test loaders...
    real=2,264  fake=2,370  total=4,634
    real=1,310  fake=7,860  total=9,170
    real=7,355  fake=63,882  total=71,237
  FoR          auc=0.0503  eer=89.44%  acc=50.95% (thr=0.99)
  WaveFake     auc=0.5291  eer=47.85%  acc=60.98% (thr=0.99)
  ASVspoof     auc=0.3136  eer=63.75%  acc=88.72% (thr=0.99)

InTheWild zero-shot eval
  real=19,963  fake=11,816  total=31,779
  InTheWild  auc=0.4429  eer=52.18%  acc=62.49% (thr=0.01)

── Generalisation retention after MLADDC fine-tuning ────────────
Dataset        Pre-FT AUC   Post-FT AUC    Delta
------------------------------------------------
  FoR             0.9908       0.0503  ▼0.9405
  WaveFake        1.0000       0.5291  ▼0.4709
  ASVspoof        0.9289       0.3136  ▼0.6153

  InTheWild (zero-shot):  auc=0.4429  eer=52.18%  acc=62.49%


In [20]:
# ==============================================================================
# NB1 Zero-shot eval on all datasets
# ==============================================================================
import gc
torch.cuda.empty_cache(); gc.collect()

cafnet_nb1 = CAFNet(n_classes=3).to(DEVICE)
cafnet_nb1.load_state_dict(torch.load(
    "/kaggle/input/models/piwoobeep/cafnet-mladdc/pytorch/default/1/cafnet_unified.pth",
    map_location=DEVICE))
cafnet_nb1.eval()

def eval_nb1_binary(model, mfcc, lfcc, chroma, labels, bs=64):
    """Eval NB1 3-class model on binary dataset. Collapses fake+ht=1."""
    all_p, all_l, all_s = [], [], []
    n = len(labels)
    mfcc   = torch.tensor(mfcc)  if not torch.is_tensor(mfcc)   else mfcc
    lfcc   = torch.tensor(lfcc)  if not torch.is_tensor(lfcc)   else lfcc
    chroma = torch.tensor(chroma) if not torch.is_tensor(chroma) else chroma
    if mfcc.dim()==3:   mfcc=mfcc.unsqueeze(1)
    if lfcc.dim()==3:   lfcc=lfcc.unsqueeze(1)
    if chroma.dim()==3: chroma=chroma.unsqueeze(1)
    with torch.no_grad():
        for i in range(0, n, bs):
            cls_main,_,_ = model(mfcc[i:i+bs].to(DEVICE),
                                  lfcc[i:i+bs].to(DEVICE),
                                  chroma[i:i+bs].to(DEVICE))
            probs = F.softmax(cls_main, dim=1).cpu().numpy()
            preds = cls_main.argmax(1).cpu().numpy()
            # P(fake) = P(class1) + P(class2)
            fake_score = probs[:,1] + probs[:,2]
            bin_preds  = (preds != 0).astype(int)
            all_p.extend(bin_preds)
            all_s.extend(fake_score)
            all_l.extend(labels[i:i+bs] if not torch.is_tensor(labels)
                         else labels[i:i+bs].numpy())
    all_p=np.array(all_p); all_l=np.array(all_l); all_s=np.array(all_s)
    bin_l = (all_l != 0).astype(int)   # 0=real, everything else=fake
    acc = accuracy_score(bin_l, all_p)
    try:
        auc = roc_auc_score(bin_l, all_s)
        eer = compute_eer((bin_l==0).astype(int), 1-all_s)
    except:
        auc=float('nan'); eer=float('nan')
    return {"acc": acc, "auc": auc, "eer": eer,
            "n_real": int((bin_l==0).sum()),
            "n_fake": int((bin_l==1).sum())}

results_nb1_zeroshot = {}

# ── MLADDC T2 (already confirmed 96.76%) ─────────────────────────────────────
print("Loading T2...")
t2 = np.load("/kaggle/input/datasets/piwoobeep/t2-processed/t2_features.npz")
r = eval_nb1_binary(cafnet_nb1,
                    t2["test_mfcc"], t2["test_lfcc"],
                    t2["test_chroma"], t2["test_labels"])
results_nb1_zeroshot["MLADDC-T2"] = r
print(f"  MLADDC-T2   acc={r['acc']*100:.2f}%  auc={r['auc']:.4f}  "
      f"eer={r['eer']*100:.2f}%  (real={r['n_real']:,} fake={r['n_fake']:,})")
del t2; gc.collect()

# ── MLADDC T3 (3-class, treat ht as fake for binary score) ───────────────────
print("Loading T3 test...")
T3_DIR = Path("/kaggle/input/datasets/piwoobeep/t3-processed/t3-processed")
t3_mfcc   = np.load(T3_DIR/"test_mfcc.npy",   mmap_mode='r')
t3_lfcc   = np.load(T3_DIR/"test_lfcc.npy",   mmap_mode='r')
t3_chroma = np.load(T3_DIR/"test_chroma.npy", mmap_mode='r')
t3_labels = np.load(T3_DIR/"test_labels.npy", mmap_mode='r')
r = eval_nb1_binary(cafnet_nb1, t3_mfcc, t3_lfcc, t3_chroma, t3_labels)
results_nb1_zeroshot["MLADDC-T3"] = r
print(f"  MLADDC-T3   acc={r['acc']*100:.2f}%  auc={r['auc']:.4f}  "
      f"eer={r['eer']*100:.2f}%  (real={r['n_real']:,} fake={r['n_fake']:,})")
del t3_mfcc, t3_lfcc, t3_chroma, t3_labels; gc.collect()

# ── FoR ───────────────────────────────────────────────────────────────────────
print("Loading FoR...")
FOR_WF_DIR = Path("/kaggle/input/datasets/piwoobeep/for-wavefake-processed")
r = eval_nb1_binary(cafnet_nb1,
                    np.load(FOR_WF_DIR/"for_test_mfcc.npy",   mmap_mode='r'),
                    np.load(FOR_WF_DIR/"for_test_lfcc.npy",   mmap_mode='r'),
                    np.load(FOR_WF_DIR/"for_test_chroma.npy", mmap_mode='r'),
                    np.load(FOR_WF_DIR/"for_test_labels.npy", mmap_mode='r'))
results_nb1_zeroshot["FoR"] = r
print(f"  FoR         acc={r['acc']*100:.2f}%  auc={r['auc']:.4f}  "
      f"eer={r['eer']*100:.2f}%  (real={r['n_real']:,} fake={r['n_fake']:,})")
gc.collect()

# ── WaveFake ──────────────────────────────────────────────────────────────────
print("Loading WaveFake...")
r = eval_nb1_binary(cafnet_nb1,
                    np.load(FOR_WF_DIR/"wf_test_mfcc.npy",   mmap_mode='r'),
                    np.load(FOR_WF_DIR/"wf_test_lfcc.npy",   mmap_mode='r'),
                    np.load(FOR_WF_DIR/"wf_test_chroma.npy", mmap_mode='r'),
                    np.load(FOR_WF_DIR/"wf_test_labels.npy", mmap_mode='r'))
results_nb1_zeroshot["WaveFake"] = r
print(f"  WaveFake    acc={r['acc']*100:.2f}%  auc={r['auc']:.4f}  "
      f"eer={r['eer']*100:.2f}%  (real={r['n_real']:,} fake={r['n_fake']:,})")
gc.collect()

# ── ASVspoof ──────────────────────────────────────────────────────────────────
print("Loading ASVspoof...")
ASV_DIR = Path("/kaggle/input/datasets/piwoobeep/asv-processed")
r = eval_nb1_binary(cafnet_nb1,
                    np.load(ASV_DIR/"asv_test_mfcc.npy",   mmap_mode='r'),
                    np.load(ASV_DIR/"asv_test_lfcc.npy",   mmap_mode='r'),
                    np.load(ASV_DIR/"asv_test_chroma.npy", mmap_mode='r'),
                    np.load(ASV_DIR/"asv_test_labels.npy", mmap_mode='r'))
results_nb1_zeroshot["ASVspoof"] = r
print(f"  ASVspoof    acc={r['acc']*100:.2f}%  auc={r['auc']:.4f}  "
      f"eer={r['eer']*100:.2f}%  (real={r['n_real']:,} fake={r['n_fake']:,})")
gc.collect()

# ── InTheWild ─────────────────────────────────────────────────────────────────
ITW_PATH = Path("/kaggle/input/datasets/piwoobeep/itw-processed")
if ITW_PATH.exists():
    print("Loading InTheWild...")
    r = eval_nb1_binary(cafnet_nb1,
                        np.load(ITW_PATH/"itw_mfcc.npy",   mmap_mode='r'),
                        np.load(ITW_PATH/"itw_lfcc.npy",   mmap_mode='r'),
                        np.load(ITW_PATH/"itw_chroma.npy", mmap_mode='r'),
                        np.load(ITW_PATH/"itw_labels.npy", mmap_mode='r'))
    results_nb1_zeroshot["InTheWild"] = r
    print(f"  InTheWild   acc={r['acc']*100:.2f}%  auc={r['auc']:.4f}  "
          f"eer={r['eer']*100:.2f}%  (real={r['n_real']:,} fake={r['n_fake']:,})")
    gc.collect()
else:
    print("InTheWild not found — skipping")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("NB1 ZERO-SHOT SUMMARY")
print("="*65)
print(f"{'Dataset':<14} {'Acc':>8} {'AUC':>8} {'EER':>8} {'Real':>8} {'Fake':>8}")
print("-"*56)
for ds, r in results_nb1_zeroshot.items():
    trained = " ◄ trained" if ds.startswith("MLADDC") else "  zero-shot"
    print(f"  {ds:<12} {r['acc']*100:>7.2f}% {r['auc']:>8.4f} "
          f"{r['eer']*100:>7.2f}% {r['n_real']:>8,} {r['n_fake']:>8,}{trained}")

Loading T2...
  MLADDC-T2   acc=96.76%  auc=0.9956  eer=3.20%  (real=5,600 fake=11,200)
Loading T3 test...
  MLADDC-T3   acc=91.09%  auc=nan  eer=nan%  (real=0 fake=16,000)
Loading FoR...
  FoR         acc=54.25%  auc=0.9289  eer=10.34%  (real=2,264 fake=2,370)
Loading WaveFake...
  WaveFake    acc=17.30%  auc=0.4948  eer=50.38%  (real=1,310 fake=7,860)
Loading ASVspoof...
  ASVspoof    acc=84.68%  auc=0.5042  eer=48.51%  (real=7,355 fake=63,882)
Loading InTheWild...
  InTheWild   acc=53.62%  auc=0.5622  eer=45.90%  (real=19,963 fake=11,816)

NB1 ZERO-SHOT SUMMARY
Dataset             Acc      AUC      EER     Real     Fake
--------------------------------------------------------
  MLADDC-T2      96.76%   0.9956    3.20%    5,600   11,200 ◄ trained
  MLADDC-T3      91.09%      nan     nan%        0   16,000 ◄ trained
  FoR            54.25%   0.9289   10.34%    2,264    2,370  zero-shot
  WaveFake       17.30%   0.4948   50.38%    1,310    7,860  zero-shot
  ASVspoof       84.68%   0.50

In [21]:
# T3 correct eval — 3 class, not binary collapse
def eval_nb1_t3(model, mfcc, lfcc, chroma, labels, bs=64):
    all_p, all_l = [], []
    mfcc=torch.tensor(mfcc).unsqueeze(1)
    lfcc=torch.tensor(lfcc).unsqueeze(1)
    chroma=torch.tensor(chroma).unsqueeze(1)
    with torch.no_grad():
        for i in range(0, len(labels), bs):
            cls_main,_,_ = model(mfcc[i:i+bs].to(DEVICE),
                                  lfcc[i:i+bs].to(DEVICE),
                                  chroma[i:i+bs].to(DEVICE))
            all_p.extend(cls_main.argmax(1).cpu().numpy())
            all_l.extend(labels[i:i+bs])
    all_p=np.array(all_p); all_l=np.array(all_l)
    acc = accuracy_score(all_l, all_p)
    print(classification_report(all_l, all_p,
          target_names=["real","fake","half-truth"]))
    return acc

T3_DIR = Path("/kaggle/input/datasets/piwoobeep/t3-processed/t3-processed")
acc = eval_nb1_t3(cafnet_nb1,
    np.load(T3_DIR/"test_mfcc.npy",   mmap_mode='r'),
    np.load(T3_DIR/"test_lfcc.npy",   mmap_mode='r'),
    np.load(T3_DIR/"test_chroma.npy", mmap_mode='r'),
    np.load(T3_DIR/"test_labels.npy", mmap_mode='r'))
print(f"T3 3-class acc: {acc*100:.2f}%")

              precision    recall  f1-score   support

        real       0.00      0.00      0.00         0
        fake       0.00      0.00      0.00         0
  half-truth       1.00      0.89      0.94     16000

    accuracy                           0.89     16000
   macro avg       0.33      0.30      0.31     16000
weighted avg       1.00      0.89      0.94     16000

T3 3-class acc: 89.19%
